In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ==========================
# CONFIG
# ==========================

FILE1 = "/content/drive/MyDrive/fp/final_predictions Krishna's input.csv"
FILE2 = "/content/drive/MyDrive/fp/final_predictions_Bhumika's input.csv"

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# ================================================================
# LEGAL FACT LAYER — GROQ EDITION
# Output: one JSON file per case in output folder
# ================================================================

# ── CELL 1: Install ──────────────────────────────────────────

#!pip install -q groq pandas tqdm
!pip install -q openai pandas tqdm

In [4]:
# ── CELL 2: Imports ──────────────────────────────────────────

import pandas as pd

import re
import json
import os
import time
import gc
import traceback
import hashlib                     # fix: collision-proof case IDs
from datetime import datetime      # fix: date normalization helpers

#from tqdm.auto import tqdm
#from groq import Groq

print("✅ Imports OK")

✅ Imports OK


In [ ]:
# ── CELL 3: Config — ONLY EDIT THIS CELL ─────────────────────
NVIDIA_API_KEY="add-key-here"
FILE_PATH      = "/content/drive/MyDrive/fp/final_predictions Krishna's input.csv"
OUTPUT_DIR = "/content/drive/MyDrive/fact_layer_v_n_final"
CHECKPOINT_DIR = "/content/drive/MyDrive/fact_layer_v_n_final/checkpoints"


PAUSE_SECS  = 5
MAX_RETRIES = 4
BATCH_SIZE  = 100
DAILY_LIMIT = 14000
RESUME      = True

# fix: Groq token-budget safety stop
MAX_TOKEN_BUDGET = 480000

os.makedirs(
   OUTPUT_DIR,
   exist_ok=True
)

os.makedirs(
    CHECKPOINT_DIR,
    exist_ok=True
)

print(
    f"✅ Config OK | Cases will be saved to → {OUTPUT_DIR}"
)

✅ Config OK | Cases will be saved to → /content/drive/MyDrive/fact_layer_v_n_final


In [6]:
## ── CELL 4: Groq client ──────────────────────────────────────


from openai import OpenAI

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=NVIDIA_API_KEY
)


SYSTEM = """
You are an expert legal information extraction system
for Indian Supreme Court judgments.

Return ONLY valid JSON.

No markdown.
No code fences.
No explanation.
No commentary.

Rules:

1. Extract only information supported by the supplied text.
2. Do not infer missing information.
3. Do not fabricate information.
4. Use null or [] when information is absent.
5. Preserve legal terminology and names accurately.
"""


# ============================================================
# JSON SCHEMAS FOR STRUCTURED OUTPUTS (strict mode)
# Every field must be listed in "required" and every object
# needs "additionalProperties": False — that's a Groq strict-mode rule,
# not optional.
# ============================================================
CASE_SCHEMA = {
    "type": "object",
    "properties": {
        "title": {"type": ["string", "null"]},
        "court": {"type": ["string", "null"]},
        "date": {"type": ["string", "null"]},
        "case_type": {"type": ["string", "null"]},
        "parties": {
            "type": "object",
            "properties": {
                "petitioner": {"type": ["string", "null"]},
                "respondent": {"type": ["string", "null"]}
            },
            "required": ["petitioner", "respondent"],
            "additionalProperties": False
        },
        "issue": {"type": ["string", "null"]},
        "facts": {"type": "array", "items": {"type": "string"}},
        "petitioner_arguments": {"type": "array", "items": {"type": "string"}},
        "respondent_arguments": {"type": "array", "items": {"type": "string"}},
        "statutes": {"type": "array", "items": {"type": "string"}},
        "citations": {"type": "array", "items": {"type": "string"}}
    },
    "required": ["title", "court", "date", "case_type",
                 "parties", "issue", "facts", "petitioner_arguments",
                 "respondent_arguments", "statutes", "citations"],
    "additionalProperties": False
}

ANALYSIS_SCHEMA = {
    "type": "object",
    "properties": {
        "reasoning_principles": {"type": "array", "items": {"type": "string"}},
        "reasoning_summary": {"type": ["string", "null"]},
        "outcome": {
            "type": ["string", "null"],
            "enum": ["Accept", "Reject", "Partially Accept", "Directions Issued", None]
        },
        "decision_details": {"type": ["string", "null"]},
        "outcome_reasoning": {"type": ["string", "null"]},
        "legal_concepts": {
        "type": "array",
        "items": {
            "type": "string",
            "enum": [
                "adverse_possession", "arbitration", "audi_alteram_partem",
                "bail_jurisprudence", "basic_structure", "beyond_reasonable_doubt",
                "burden_of_proof", "circumstantial_evidence", "consumer_protection",
                "delegated_legislation", "doctrine_of_merger",
                "doctrine_of_severability", "double_jeopardy", "due_process",
                "dying_declaration", "estoppel", "fair_trial", "force_majeure",
                "fundamental_rights", "good_faith", "habeas_corpus",
                "hearsay_evidence", "hostile_witness", "judicial_review",
                "legitimate_expectation", "limitation", "locus_standi",
                "mala_fides", "mens_rea", "natural_justice", "preventive_detention",
                "precedent", "promissory_estoppel", "property_law",
                "public_interest", "quasi_judicial", "reasonable_doubt",
                "res_judicata", "right_against_self_incrimination",
                "right_to_equality", "right_to_life", "right_to_privacy",
                "rule_against_bias", "rule_of_law", "separation_of_powers",
                "service_law", "specific_performance", "stare_decisis", "tax_law",
                "ultra_vires", "unjust_enrichment", "vicarious_liability",
                "writ_jurisdiction"
            ]
        }
    },
        "case_summary": {"type": ["string", "null"]}
    },
    "required": ["reasoning_principles", "reasoning_summary", "outcome",
                 "decision_details", "outcome_reasoning", "legal_concepts",
                 "case_summary"],
    "additionalProperties": False
}

# ============================================================
# TOKEN ACCOUNTING
# ============================================================

total_tokens_used = 0


# ============================================================
# DAILY LIMIT EXCEPTION
# ============================================================

class DailyLimitReached(Exception):
    pass


# ============================================================
# OUTCOME NORMALIZATION
# ============================================================

OUTCOME_MAP = {

    # ACCEPT
    "accept": "Accept",
    "accepted": "Accept",
    "allow": "Accept",
    "allowed": "Accept",
    "granted": "Accept",
    "quashed": "Accept",
    "set aside": "Accept",

    # REJECT
    "reject": "Reject",
    "rejected": "Reject",
    "dismiss": "Reject",
    "dismissed": "Reject",
    "upheld": "Reject",

    # PARTIAL
    "partial": "Partially Accept",
    "partially accept": "Partially Accept",
    "partially accepted": "Partially Accept",
    "partially allowed": "Partially Accept",
    "partly allowed": "Partially Accept",
    "modified": "Partially Accept",

    # SPECIAL
    "withdrawn": "Withdrawn",

    # DIRECTIONS ISSUED
    "directions issued": "Directions Issued",
    "direction issued": "Directions Issued",
    "directions given": "Directions Issued",
    "continuing mandamus": "Directions Issued",
}


def normalize_outcome(outcome):

    if not outcome:
        return None

    outcome = str(outcome).strip().lower()

    if outcome in OUTCOME_MAP:
        return OUTCOME_MAP[outcome]

    if any(
        x in outcome
        for x in [
            "partially allowed",
            "partly allowed",
            "modified",
            "partial"
        ]
    ):
        return "Partially Accept"

    if any(
        x in outcome
        for x in [
            "allowed",
            "accepted",
            "granted",
            "quashed",
            "set aside"
        ]
    ):
        return "Accept"

    if any(
        x in outcome
        for x in [
            "dismissed",
            "rejected",
            "upheld"
        ]
    ):
        return "Reject"

    if any(x in outcome for x in ["directions issued", "direction issued", "continuing mandamus", "directions given"]):
        return "Directions Issued"

    return None


# ============================================================
# RATE LIMIT HANDLER
# ============================================================

def _handle_rate_limit(exc, retry_no):

    err = str(exc).lower()

    print("\n" + "="*80)
    print("EXCEPTION TYPE:", type(exc).__name__)
    print("EXCEPTION:")
    print(exc)
    print("="*80)

    err = str(exc).lower()

    # --------------------------------------------------------
    # DAILY TOKEN LIMIT
    # --------------------------------------------------------

    if (
        "tokens per day" in err
        or "daily token" in err
        or "daily limit" in err
        or "per day" in err
    ):

        print(
            "\n🛑 Daily Groq token limit reached."
            "\nResume tomorrow."
        )

        raise DailyLimitReached(err)

    # --------------------------------------------------------
    # MINUTE TOKEN LIMIT
    # --------------------------------------------------------

    if (
        "tokens per minute" in err
        or "rate_limit_tokens" in err
        or "rate_limit" in err
        or "429" in err
    ):

        wait = 65

        print(
            f"\n⏳ Minute limit reached."
            f"\nWaiting {wait} seconds..."
        )

        time.sleep(wait)

        return

    # --------------------------------------------------------
    # UNKNOWN ERROR
    # --------------------------------------------------------

    print("\nUnknown exception encountered.")
    raise exc


# ============================================================
# GROQ CALL
# ============================================================

def call_groq(prompt: str, max_tokens=1600, json_schema=None, schema_name="extraction"):

    """
    Returns:

        (parsed_json, "groq")

    OR

        (None, "regex_fallback")
    """

    global total_tokens_used

    if total_tokens_used >= MAX_TOKEN_BUDGET:

        raise DailyLimitReached(
            "Local token budget exhausted"
        )

    prompt_tokens_estimate = None  # learned the first time we hit a 413 for this call

    attempts = [

        # Normal extraction
        (
            0.05,
            ""
        ),

        # JSON repair pass
        (
            0.0,
            (
                "Your previous response was not valid JSON.\n"
                "Return ONLY a valid JSON object.\n"
                "Start with { and end with }.\n\n"
            )
        ),
    ]

    for attempt_index, (temperature, prefix) in enumerate(attempts):

        for retry in range(MAX_RETRIES):

            try:

                resp = client.chat.completions.create(
                  model="openai/gpt-oss-20b",
                  messages=[
                    {"role": "system", "content": SYSTEM},
                    {"role": "user", "content": prefix + prompt}
                  ],
                  temperature=temperature,
                  max_tokens=max_tokens,
                  reasoning_effort="low",              # change 1 — less budget spent "thinking"
                  response_format={                    # change 2 — forces schema-valid JSON
                    "type": "json_schema",
                    "json_schema": {
                      "name": schema_name,
                      "strict": True,
                      "schema": json_schema
                      }
                  } if json_schema else None
                )

                try:
                    total_tokens_used += (
                        resp.usage.total_tokens
                    )
                except:
                    pass

                content = resp.choices[0].message.content

                if content is None:
                    raise RuntimeError(
                        f"Model returned no content. Response: {resp}"
                    )

                raw = content.strip()

                raw = re.sub(
                    r"^```(?:json)?\s*",
                    "",
                    raw
                )

                raw = re.sub(
                    r"\s*```$",
                    "",
                    raw
                )

                parsed = json.loads(raw)

                time.sleep(PAUSE_SECS)

                return parsed, "groq"

            except json.JSONDecodeError:

                if attempt_index == 0:
                    break

                return (
                    None,
                    "regex_fallback"
                )

            # NEW:
            except DailyLimitReached:
                    raise
            except Exception as e:
                err_str = str(e).lower()

                    # 413 — prompt + max_tokens exceeds the per-minute ceiling.
                    # Shrink by exactly the overage, not a blind percentage —
                    # over-shrinking starves the JSON of room to finish (see below).
                if "request too large" in err_str or "413" in err_str:
                    m = re.search(r"limit (\d+), requested (\d+)", err_str)
                    old_mt = max_tokens
                    if m:
                        limit, requested = int(m.group(1)), int(m.group(2))
                        prompt_tokens_estimate = requested - max_tokens
                        overage = requested - limit
                        max_tokens = max(600, max_tokens - overage - 50)
                    else:
                        max_tokens = max(600, int(max_tokens * 0.6))
                    print(
                        f"\n⚠️ Request too large (413) — prompt is too big "
                        f"for the {old_mt}-token budget. "
                        f"Shrinking max_tokens {old_mt} → {max_tokens} and retrying immediately."
                    )
                    continue

                    # 400 — valid JSON but cut off before required fields were
                    # written, because a previous 413-shrink left too little room.
                    # Grow back, but stay under the same TPM ceiling we just learned.
                if "does not match the expected schema" in err_str and "missing properties" in err_str:
                    old_mt = max_tokens
                    if prompt_tokens_estimate is not None:
                        safe_ceiling = max(600, 7900 - prompt_tokens_estimate)
                        max_tokens = min(int(max_tokens * 1.4), safe_ceiling)
                    else:
                        max_tokens = min(int(max_tokens * 1.4), 4500)
                    print(
                        f"\n⚠️ Response cut off before all required fields were written "
                        f"(ran out of the {old_mt}-token budget mid-JSON). "
                        f"Raising max_tokens {old_mt} → {max_tokens} and retrying."
                    )
                    continue

                _handle_rate_limit(
                    e,
                    retry
                )

    return (
        None,
        "regex_fallback"
    )


print("✅ Groq client ready")

✅ Groq client ready


In [7]:
# ── CELL 5A: Case Extraction Prompt ──────────────────────────

def build_case_prompt(
    full_text,
    facts,
    petitioner_args,
    respondent_args,
    title_hint,
    year
):
    return f"""
You are an expert legal information extraction system for
Indian Supreme Court judgments.

Return ONLY valid JSON.
No markdown.
No explanation outside JSON.

IMPORTANT RULES:

1. Extract information ONLY from the supplied judgment.
2. Do NOT infer missing information.
3. Do NOT fabricate information.
4. If information is absent, return null or [].
5. Use Indian legal terminology.
6. Preserve names and legal provisions accurately.

------------------------------------------------------------
TEXT ARTIFACTS
------------------------------------------------------------

The source text was extracted from a PDF and may contain artifacts
from page headers/footers accidentally embedded mid-sentence, usually
a repetition of the case title and date. If you see a fragment like
this interrupting an otherwise coherent sentence, silently remove
ONLY that injected fragment and reconstruct the real sentence. Do not
remove content that is genuinely part of the sentence.

------------------------------------------------------------
ARGUMENTS — LABEL VERIFICATION
------------------------------------------------------------

The FACTS / PETITIONER ARGUMENTS / RESPONDENT ARGUMENTS segments
below were pre-labeled by an automatic classifier that sometimes
confuses petitioner and respondent arguments. Do NOT blindly trust
these labels. Read each argument's actual content and determine
which side it truly supports, using cues such as "the appellant
contends...", "the respondent submits...", or which party's position
the argument defends or attacks. If an argument appears under the
wrong heading, move it to the correct side in your output. If you
cannot confidently tell which side an argument belongs to, omit it
rather than guessing.

CASE YEAR:
{year}

FILENAME HINT:
{title_hint}

------------------------------------------------------------
PRE-SEGMENTED SECTIONS (for reference — verify, don't blindly trust)
------------------------------------------------------------

FACTS SEGMENT:
{facts[:3000]}

PETITIONER ARGUMENTS SEGMENT:
{petitioner_args[:2000]}

RESPONDENT ARGUMENTS SEGMENT:
{respondent_args[:2000]}

------------------------------------------------------------
FULL JUDGMENT TEXT (use this for parties/date/statutes/citations
— those can appear anywhere, not just in the segments above)
------------------------------------------------------------

{full_text[:12000]}

Return exactly this JSON structure:

{{
    "title": null,
    "court": null,
    "date": null,
    "case_type": null,

    "parties": {{
        "petitioner": null,
        "respondent": null
    }},

    "issue": null,

    "facts": [],

    "petitioner_arguments": [],

    "respondent_arguments": [],

    "statutes": [],

    "citations": []
}}


PARTIES:

Identify the actual petitioner/appellant and respondent
from the judgment.

Do not infer parties from unrelated text.

FACTS:

Facts must describe real-world events that gave rise
to the dispute.

Do NOT include:
- filing history
- procedural history
- lower court proceedings
- appeal chronology
- court reasoning

ISSUE:

Return ONE precise legal question that the Court answered.

ARGUMENTS:

Petitioner arguments must come only from petitioner/appellant material.

Respondent arguments must come only from respondent material.

If respondent arguments are absent, return [].

STATUTES:

Extract specific statutory or constitutional provisions
actually mentioned or applied in the judgment.

Examples:
- Section 302 IPC
- Article 14 of the Constitution
- Order 7 Rule 11 CPC

Do not invent an Act name.Do not guess or infer the parent Act name for a bare section number based
on assumption or pattern-matching. Only attach an Act name if it is
explicitly stated in the judgment text, or if the context makes the Act
unambiguous (e.g., a motor accident compensation claim under "Section
166" is virtually always the Motor Vehicles Act, 1988 — never the
Insurance Act). If you are not confident of the correct Act, leave the
provision as a bare section number rather than attributing it to the
wrong Act.

Do not include a constitutional Article or statutory section merely
because the number appears somewhere in the source text (e.g. in a
citation, docket number, or unrelated numbered clause). Only include a
provision if the judgment text discusses or applies it substantively in
connection with the legal issue being decided. If you are not certain a
provision is actually relevant to this case's reasoning, omit it.

CITATIONS:

Extract case citations and reported case references
actually appearing in the judgment.

Examples:
- AIR 1994 SC 736
- (2008) 17 SCC 703

Do not invent citations.

DATE:

Return the judgment date if explicitly present.
Use ISO format YYYY-MM-DD when it can be determined
directly from the judgment. Otherwise return null.

CASE TYPE:

Return the most appropriate case type only when supported
by the judgment. Otherwise return null.
"""

In [8]:
# ── CELL 5B: Legal Analysis Prompt ───────────────────────────

def build_analysis_prompt(
    facts,
    reasoning,
    decision,
    full_text
):
    return f"""
You are an expert legal reasoning extraction system for
Indian Supreme Court judgments.

Return ONLY valid JSON.
No markdown.
No explanation outside JSON.

IMPORTANT:

1. Use ONLY information supported by the judgment.
2. Do NOT infer.
3. Do NOT fabricate.
4. Do NOT create hypothetical reasoning.
5. If information is absent, return null or [].

RELEVANT FACTS:
{facts[:3000]}

COURT REASONING:
{reasoning[:5000]}

COURT DECISION:
{decision[:3000]}

Return exactly:

{{
    "reasoning_principles": [],

    "reasoning_summary": null,

    "outcome": null,

    "decision_details": null,

    "outcome_reasoning": null,

    "legal_concepts": [],

    "case_summary": null
}}

REASONING PRINCIPLES:

Extract legal doctrines, constitutional doctrines,
statutory interpretations and legal tests that were
actually DISCUSSED AND APPLIED by the Court.

Examples:

- Basic Structure Doctrine
- Proportionality Test
- Natural Justice
- Doctrine of Severability
- Reasonable Classification

Do not include generic observations.

REASONING SUMMARY:

Briefly explain how the Court reasoned from the relevant
law to its conclusion.

OUTCOME:

MUST be exactly one of:

Accept
Reject
Partially Accept
Directions Issued

Mapping:

Allowed / Granted / Quashed -> Accept
Dismissed / Rejected / Upheld -> Reject
Partly Allowed / Partially Allowed -> Partially Accept

Use "Directions Issued" ONLY if there is no winning/losing party in the
ordinary sense — e.g. a PIL where the Court supervises an authority's
ongoing compliance across time, with no single appeal being allowed or
dismissed. The test: is there a specific party who won or lost THIS
appeal? If yes, classify normally as Accept/Reject/Partially Accept,
even if the reasoning is complex or the relief is "go redo this
correctly" rather than a final award.

Two common traps:
- Almost every judgment's operative order contains the word "direct"
  (e.g. "we direct that the sale proceeds be transferred to X"). This
  alone does NOT make it "Directions Issued."
- A remand — the Court sets aside a lower order and sends the matter
  back for reconsideration, even with a compliance deadline attached —
  is still Accept/Partially Accept for the party who won the remand,
  not "Directions Issued."

DECISION DETAILS:

State the specific order actually passed by the Court.

OUTCOME REASONING:

Explain in one sentence what the Court ordered and to whom.

LEGAL CONCEPTS:

Select ONLY from this fixed list of legal concept IDs — do not invent
new concept names, even if a more specific term would fit better:

adverse_possession, arbitration, audi_alteram_partem, bail_jurisprudence,basic_structure,
beyond_reasonable_doubt, burden_of_proof, circumstantial_evidence,
consumer_protection, delegated_legislation, doctrine_of_merger,
doctrine_of_severability, double_jeopardy, due_process, dying_declaration,
estoppel, fair_trial, force_majeure, fundamental_rights, good_faith,
habeas_corpus, hearsay_evidence, hostile_witness, judicial_review,
legitimate_expectation, limitation, locus_standi, mala_fides, mens_rea,
natural_justice, preventive_detention, precedent, promissory_estoppel,
property_law, public_interest, quasi_judicial, reasonable_doubt,
res_judicata, right_against_self_incrimination, right_to_equality,
right_to_life, right_to_privacy, rule_against_bias, rule_of_law, separation_of_powers,
service_law, specific_performance, stare_decisis, tax_law, ultra_vires,
unjust_enrichment, vicarious_liability, writ_jurisdiction

Only select a concept if it is a genuine legal basis for the Court's
reasoning — not merely a term that sounds topically related to the
subject matter of the case.

Return only the ones actually applicable. If none of the listed
concepts genuinely apply, return [] rather than the closest match.

CASE SUMMARY:

Briefly explain the legal significance of the case,
based only on the judgment.
"""

CELL 6 — PART 1

In [9]:
# ── CELL 6: Regex helpers ────────────────────────────────────

# ============================================================
# DATE HELPERS
# ============================================================

_MONTHS = {
    "january": 1,
    "february": 2,
    "march": 3,
    "april": 4,
    "may": 5,
    "june": 6,
    "july": 7,
    "august": 8,
    "september": 9,
    "october": 10,
    "november": 11,
    "december": 12,
}

_DATE_PATTERNS = [

    (
        r"(\d{1,2})(?:st|nd|rd|th)\s+([A-Za-z]+),?\s+(\d{4})",
        "dmy_ordinal"
    ),

    (
        r"([A-Za-z]+)\s+(\d{1,2}),?\s+(\d{4})",
        "mdy_text"
    ),

    (
        r"DATED?\s+this\s+(\d{1,2})(?:st|nd|rd|th)\s+day\s+of\s+([A-Za-z]+)\s+(\d{4})",
        "formal"
    ),

    (
        r"(\d{2})[./\-](\d{2})[./\-](\d{4})",
        "dmy_numeric"
    ),

    (
        r"DATE\s+OF\s+JUDGMENT[:\s]+(\d{2}/\d{2}/\d{4})",
        "labeled"
    ),
]


def _to_iso(groups, fmt):

    try:

        if fmt in ("dmy_ordinal", "formal"):

            d, m, y = groups

            mn = _MONTHS.get(
                m.lower()
            )

            if not mn:
                return None

            return f"{y}-{mn:02d}-{int(d):02d}"

        if fmt == "mdy_text":

            m, d, y = groups

            mn = _MONTHS.get(
                m.lower()
            )

            if not mn:
                return None

            return f"{y}-{mn:02d}-{int(d):02d}"

        if fmt == "dmy_numeric":

            d, m, y = groups

            return f"{y}-{m}-{d}"

        if fmt == "labeled":

            d, m, y = groups[0].split("/")

            return f"{y}-{m}-{d}"

    except:
        return None

    return None


# ============================================================
# STATUTES
# ============================================================

_STATUTE_PATTERNS = [

    r"(?:Article|Art\.)\s*\d+[A-Z]?(?:\([a-z0-9]\))*(?:\s+of\s+(?:the\s+)?Constitution)?",

    r"[Ss]ection\s+\d+[A-Z]?\s+of\s+(?:the\s+)?[A-Z][^\n,;\.]{3,60}?(?:Act|Code|Ordinance)",

    r"(?:IPC|CrPC|CPC)\s+(?:[Ss]ection\s+)?\d+[A-Z]?",

    # fix
    r"[Uu]/[Ss]\s*\.?\s*\d+[A-Z]?(?:\([a-z0-9]+\))?\s+(?:IPC|CrPC|CPC|IEA)",

    # fix
    r"[Ss]\.?\s*\d+[A-Z]?(?:\([a-z0-9]+\))?\s+(?:IPC|CrPC|CPC|IEA)",

    # fix
    r"[Ss]ec(?:tion)?\.?\s*\d+[A-Z]?(?:\([a-z0-9]+\))*",

    # fix
    r"[Aa]rticle\s+\d+[A-Z]?\s*/\s*\d+[A-Z]?",

    # fix
    r"[Oo]rder\s+(?:[IVXLC]+|\d+)\s+[Rr]ule\s+\d+[A-Z]?\s+(?:CPC|CrPC)",

    # fix
    r"[Ss]chedule\s+(?:[IVXLC]+|\d+)\s+[Ee]ntry\s+\d+[A-Z]?",
]


def roman_to_int(s):

    vals = {
        "I":1,
        "V":5,
        "X":10,
        "L":50,
        "C":100
    }

    total = 0
    prev = 0

    for ch in reversed(s.upper()):

        cur = vals.get(ch,0)

        if cur < prev:
            total -= cur
        else:
            total += cur

        prev = cur

    return total


def _norm_statute(raw):

    raw = raw.strip()

    m = re.search(
        r"[Aa]rticle\s+(\d+[A-Z]?)\s*/\s*(\d+[A-Z]?)",
        raw
    )

    if m:

        return [
            f"Constitution_Art_{m.group(1)}",
            f"Constitution_Art_{m.group(2)}",
        ]

    m = re.search(
        r"[Aa]rticle\s+(\d+[A-Z]?(?:\([a-z0-9]+\))*)",
        raw
    )

    if m:
        return f"Constitution_Art_{m.group(1)}"

    m = re.search(
        r"(IPC|CrPC|CPC|IEA)\s+(?:[Ss]ection\s+)?(\d+[A-Z]?)",
        raw,
        re.I
    )

    if m:
        return f"{m.group(1)}_S{m.group(2)}"

    m = re.search(
        r"[Uu]/[Ss]\s*\.?\s*(\d+[A-Z]?)",
        raw
    )

    if m:
        return f"IPC_S{m.group(1)}"

    m = re.search(
        r"[Ss]\.?\s*(\d+[A-Z]?)",
        raw
    )

    if m:
        return f"IPC_S{m.group(1)}"

    m = re.search(
        r"[Ss]ec(?:tion)?\.?\s*(\d+[A-Z]?(?:\([a-z0-9]+\))*)",
        raw
    )

    if m:
        return f"Section_{m.group(1)}"

    m = re.search(
        r"[Oo]rder\s+([IVXLC]+|\d+)\s+[Rr]ule\s+(\d+[A-Z]?)",
        raw,
        re.I
    )

    if m:

        order = m.group(1)

        if order.isalpha():
            order = roman_to_int(order)

        return f"CPC_Order{order}_Rule{m.group(2)}"

    m = re.search(
        r"[Ss]chedule\s+([IVXLC]+|\d+)\s+[Ee]ntry\s+(\d+[A-Z]?)",
        raw,
        re.I
    )

    if m:

        sch = m.group(1)

        if sch.isalpha():
            sch = roman_to_int(sch)

        return (
            f"Constitution_Schedule{sch}"
            f"_Entry{m.group(2)}"
        )

    return None


def extract_statutes(text):
    seen = set()
    out = []
    for pat in _STATUTE_PATTERNS:
        for m in re.finditer(pat, text, re.I):
            raw = m.group(0)

            # Bare "Article N" with no "of the Constitution" suffix needs
            # "constitution" somewhere nearby, or we drop it as unreliable.
            if re.match(r"(?:article|art\.)", raw, re.I) and "constitution" not in raw.lower():
                window = text[max(0, m.start()-60): m.end()+60].lower()
                if "constitution" not in window:
                    continue

            sid = _norm_statute(raw)
            if not sid:
                continue
            if isinstance(sid, list):
                for x in sid:
                    if x not in seen:
                        seen.add(x)
                        out.append(x)
            else:
                if sid not in seen:
                    seen.add(sid)
                    out.append(sid)
    return out


# ============================================================
# CITATIONS
# ============================================================

_CITATION_PATTERNS = [

    r"\(\d{4}\)\s+\d+\s+SCC\s+\d+",

    r"\d{4}\s*\(\s*\d+\s*\)\s*SCC\s+\d+",

    r"\d{4}\s+Supp(?:l(?:ement)?)?\s+SCC\s+\d+",

    r"JT\s+\d{4}\s*\(\s*\d+\s*\)\s*SC\s+\d+",

    r"MANU/(?:SC|HC)/\d{1,6}/\d{4}",

    r"W\.?P\.?\s*\([A-Z]+\)\s*[Nn]o\.?\s*\d+\s*/\s*\d{4}",

    r"(?:Civil|Criminal)\s+Appeal\s+[Nn]o\.?\s*\d+\s+of\s+\d{4}",

    r"AIR\s+\d{4}\s+SC(?:\s+\d+)?",

    r"\d{4}\s+SCR\s+\d+",
]


def extract_citations(text):

    found = []

    for pat in _CITATION_PATTERNS:

        for m in re.finditer(
            pat,
            text,
            re.I
        ):

            c = re.sub(
                r"\s+",
                " ",
                m.group(0).strip()
            )

            if c not in found:
                found.append(c)

    return found

CELL 6 — PART 2 (Metadata, Parties, Concepts, Outcome)

In [10]:

# ============================================================
# METADATA
# ============================================================

def extract_metadata(text):

    hdr = text[:3000]

    date = None

    for pat, fmt in _DATE_PATTERNS:

        m = re.search(
            pat,
            hdr,
            re.I
        )

        if m:

            date = _to_iso(
                m.groups(),
                fmt
            )

            if date:
                break

    court = "Supreme Court of India"

    cm = re.search(
        r"(High Court of [A-Za-z\s]+)",
        hdr,
        re.I
    )

    if cm:

        court = (
            cm.group(1)
            .strip()
            .title()
        )


    return {
        "date": date,
        "court": court,
    }


# ============================================================
# PARTY EXTRACTION
# ============================================================

def _extract_party_from_text(
    text,
    role
):

    m = re.search(
        rf"(?:{role}|appellant)[,\s]+([A-Z][A-Za-z\s\.]+?)(?:,|\s+(?:has|have|filed|submits|is|was)\b)",
        text[:500]
    )

    if m:
        return m.group(1).strip().title()

    return None


def extract_parties(
    filename,
    facts_text=""
):

    fn = re.sub(
        r"\.pdf$",
        "",
        filename,
        flags=re.I
    )

    # fix
    fn = re.sub(
        r"_{2,}",
        "_",
        fn
    )

    # fix
    fn = re.sub(
        r"\bM_[Ss]\b",
        "M/s",
        fn
    )

    # ===================================================
    # In Re
    # ===================================================

    if re.match(
        r"In_Re_",
        fn,
        re.I
    ):

        name = (
            fn[6:]
            .replace("_"," ")
            .strip()
            .title()
        )

        return (
            f"In Re {name}",
            None
        )

    # ===================================================
    # Suo Motu
    # ===================================================

    if re.match(
        r"Suo_Motu",
        fn,
        re.I
    ):

        return (
            "Court (Suo Motu)",
            None
        )

    # ===================================================
    # Standard vs
    # ===================================================

    m = re.search(
        r"(.+?)_vs?_(.+?)(?:__on_|_on_|\d{4}|$)",
        fn,
        re.I
    )

    if m:

        pet = (
            m.group(1)
            .replace("_"," ")
            .strip()
            .title()
        )

        res = (
            m.group(2)
            .replace("_"," ")
            .strip()
            .title()
        )

        pet = pet.replace(
            "M S ",
            "M/s "
        )

        return pet, res

    # ===================================================
    # Text fallback
    # ===================================================

    pet = _extract_party_from_text(
        facts_text,
        "petitioner"
    )

    res = _extract_party_from_text(
        facts_text,
        "respondent"
    )

    return pet, res


# ============================================================
# LEGAL CONCEPTS
# ============================================================

_CONCEPTS = {

    "natural_justice":
        ["natural justice","audi alteram partem"],

    "fundamental_rights":
        ["fundamental rights","article 14","article 19","article 21"],

    "due_process":
        ["due process","procedure established by law"],

    "mens_rea":
        ["mens rea","guilty mind"],

    "res_judicata":
        ["res judicata"],

    "habeas_corpus":
        ["habeas corpus"],

    "doctrine_of_severability":
        ["doctrine of severability","severable"],

    "ultra_vires":
        ["ultra vires"],

    "locus_standi":
        ["locus standi"],

    "writ_jurisdiction":
        ["certiorari","mandamus","writ jurisdiction"],

    "precedent":
        ["ratio decidendi","obiter dicta","stare decisis"],

    "fair_trial":
        ["fair trial","right to counsel"],

    "preventive_detention":
        ["preventive detention"],

    "promissory_estoppel":
        ["promissory estoppel","legitimate expectation"],

    "vicarious_liability":
        ["vicarious liability"],

    # NEW

    "beyond_reasonable_doubt":
        ["beyond reasonable doubt","benefit of doubt"],

    "circumstantial_evidence":
        ["circumstantial evidence","last seen theory"],

    "bail_jurisprudence":
        ["anticipatory bail","default bail"],

    "limitation":
        ["limitation period","cause of action"],

    "specific_performance":
        ["specific performance","breach of contract"],

    "tax_law":
        ["tax evasion","assessment order"],

    "service_law":
        ["service rules","departmental proceedings"],

    "property_law":
        ["adverse possession","title deed"],
        "double_jeopardy":
        ["double jeopardy"],

    "hostile_witness":
        ["hostile witness"],

    "dying_declaration":
        ["dying declaration"],

    "burden_of_proof":
        ["burden of proof"],

    "arbitration":
        ["arbitration", "arbitral award"],

    "consumer_protection":
        ["consumer protection", "consumer dispute"],
}


def extract_concepts(text):

    t = text.lower()

    return [
        k
        for k, kws in _CONCEPTS.items()
        if any(
            kw in t
            for kw in kws
        )
    ]


# ============================================================
# CASE ID
# ============================================================

def make_case_id(
    year,
    filename
):

    h = hashlib.md5(
        filename.encode()
    ).hexdigest()[:12]

    return f"{year}_{h}"


_DATE_SUFFIX_RE = re.compile(
    r"_on_(\d{1,2})_([A-Za-z]+)_(\d{4})(?:_\d+)?$",
    re.I
)

_MONTHS = {m.lower(): i for i, m in enumerate(
    ["January","February","March","April","May","June",
     "July","August","September","October","November","December"], start=1)}

def extract_date_from_filename(filename):
    fn = re.sub(r"\.pdf$", "", filename, flags=re.I)
    m = _DATE_SUFFIX_RE.search(fn)
    if not m:
        return None
    day, month_name, year = m.groups()
    month = _MONTHS.get(month_name.lower())
    if not month:
        return None
    return f"{int(year):04d}-{month:02d}-{int(day):02d}"

def title_from_filename(filename):
    fn = re.sub(r"\.pdf$", "", filename, flags=re.I)
    fn = re.sub(r"__on_.+$", "", fn)      # old double-underscore variant
    fn = _DATE_SUFFIX_RE.sub("", fn)      # "_on_25_April_2000" or "_on_25_April_2000_1"
    fn = re.sub(r"_\d+$", "", fn)         # any remaining trailing "_1" dedup digit
    return fn.replace("_", " ").strip().title()


# ============================================================
# OUTCOME
# ============================================================

def regex_outcome(decision):

    t = decision.lower()

    # --------------------------------------------------
    # withdrawn
    # --------------------------------------------------

    if re.search(
        r"dismissed\s+as\s+withdrawn|withdrawn\s+by\s+(?:the\s+)?petitioner",
        t
    ):
        return "Unknown"

    # --------------------------------------------------
    # respondent objection dismissed
    # --------------------------------------------------

    if re.search(
        r"(?:respondent'?s?|objection|preliminary\s+objection).{0,40}?dismissed",
        t
    ):
        return "Accept"

    # --------------------------------------------------
    # strongest signals
    # --------------------------------------------------

    if re.search(
        r"(?:petition|appeal|application)\s+(?:is\s+)?(?:partly|partially)\s+allowed",
        t
    ):
        return "Partially Accept"

    if re.search(
        r"(?:petition|appeal|application)\s+(?:is\s+)?(?:allowed|accepted|granted)",
        t
    ):
        return "Accept"

    if re.search(
        r"(?:petition|appeal|application)\s+(?:is\s+)?dismissed",
        t
    ):
        return "Reject"

    # --------------------------------------------------
    # secondary signals
    # --------------------------------------------------

    if any(
        x in t
        for x in [
            "partly allowed",
            "partially allowed",
            "allowed in part",
            "modified"
        ]
    ):
        return "Partially Accept"

    if any(
        x in t
        for x in [
            "quashed",
            "set aside",
            "acquitted"
        ]
    ):
        return "Accept"

    if any(
        x in t
        for x in [
            "dismissed",
            "fails",
            "no merit"
        ]
    ):
        return "Reject"

    return "Unknown"

def safe_filename(case_id):

    return (
        re.sub(
            r"[^\w\-]",
            "_",
            case_id
        )[:120]
        + ".json"
    )


print("✅ Helpers ready")

✅ Helpers ready


CELL 7 — Load CSV

In [11]:
# ── CELL 7: Load CSV ─────────────────────────────────────────

print("Loading CSV...")

df = pd.read_csv(
    FILE_PATH,
    dtype=str
)

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
)

# fix: confidence column detection
HAS_CONFIDENCE = (
    "confidence" in df.columns
)

df["year"] = pd.to_numeric(
    df["year"],
    errors="coerce"
)

# fix: lowercase filename for collision-proof case grouping
df["_casekey"] = (
    df["filename"]
    .fillna("unknown")
    .str.lower()
    + "__"
    + df["year"].astype(str)
)

all_keys = (
    df["_casekey"]
    .unique()
    .tolist()
)

print(
    f"✅ {len(df):,} rows | "
    f"{len(all_keys):,} unique cases"
)

if HAS_CONFIDENCE:
    print("✅ confidence column detected")
else:
    print(
        "⚠️ confidence column not found "
        "(confidence filtering disabled)"
    )

Loading CSV...
✅ 3,684,243 rows | 26,688 unique cases
✅ confidence column detected


CELL 8 — Checkpoint

In [12]:
# ── CELL 8: Checkpoint ───────────────────────────────────────

CKPT_FILE = os.path.join(
    CHECKPOINT_DIR,
    "done_keys.json"
)

ERROR_FILE = os.path.join(
    CHECKPOINT_DIR,
    "errors.json"
)


def load_done() -> set:

    if (
        RESUME
        and os.path.exists(CKPT_FILE)
    ):

        try:

            with open(
                CKPT_FILE,
                "r",
                encoding="utf-8"
            ) as f:

                keys = set(
                    json.load(f)
                )

            print(
                f"↩ Resuming — "
                f"{len(keys):,} cases already completed"
            )

            return keys

        except Exception as e:

            print(
                "⚠️ Failed to load checkpoint:",
                e
            )

            return set()

    return set()


def save_done(keys: set):

    try:

        with open(
            CKPT_FILE,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                list(keys),
                f,
                indent=2
            )

    except Exception as e:

        print(
            "⚠️ Failed to save checkpoint:",
            e
        )


def save_errors(errors):

    try:

        with open(
            ERROR_FILE,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                errors,
                f,
                indent=2,
                ensure_ascii=False
            )

    except Exception as e:

        print(
            "⚠️ Failed to save error log:",
            e
        )


def save_case(record: dict):

    """
    Save one JSON file per case.
    """

    fname = safe_filename(
        record["case_id"]
    )

    path = os.path.join(
        OUTPUT_DIR,
        fname
    )

    with open(
        path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            record,
            f,
            indent=2,
            ensure_ascii=False
        )


# fix: helper used when daily token limit reached
def save_checkpoint_and_stop(
    done_keys,
    errors=None
):

    print(
        "\n💾 Saving checkpoint..."
    )

    save_done(done_keys)

    if errors:
        save_errors(errors)

    print(
        "✅ Checkpoint saved."
    )


print("✅ Checkpoint ready")

✅ Checkpoint ready


In [13]:
# ── MONITORING HELPERS — Tier 1, concepts diagnostic, Tier 2 ──

import re, json, glob, random
from collections import Counter
import pandas as pd

MONITORING_DIR = os.path.join(OUTPUT_DIR, "monitoring")
os.makedirs(MONITORING_DIR, exist_ok=True)

_NUM_EXTRACT_RE = re.compile(r"\d{1,3}[A-Za-z]?")

def reconstruct_full_text(casekey, df):
    group = df[df["_casekey"] == casekey]
    lbl_map = {}
    for _, row in group.iterrows():
        if HAS_CONFIDENCE:
            try:
                conf = float(row.get("confidence", 1.0))
            except:
                conf = 1.0
            if conf < 0.65:
                continue
        label = str(row["predicted_label"]).strip()
        txt = str(row["text"]).strip()
        lbl_map.setdefault(label, []).append(txt)

    def join(*labels):
        vals = []
        for lbl in labels:
            vals.extend(lbl_map.get(lbl, []))
        return smart_join(vals)

    return smart_join([
        join("Facts"), join("Arguments of Petitioner"),
        join("Arguments of Respondent"), join("Reasoning"), join("Decision")
    ])


# ============================================================
# TIER 1 — statute/citation grounding check
# ============================================================

def run_tier1_audit(output_dir, df, checkpoint_label):
    flagged = []
    checked = 0

    for f in glob.glob(os.path.join(output_dir, "*.json")):
        try:
            d = json.load(open(f))
        except Exception:
            continue
        case_id = d.get("case_id")
        filename = d.get("filename")
        year = d.get("year")
        if not filename or year is None:
            continue

        casekey = f"{str(filename).lower()}__{year}"
        full_text = reconstruct_full_text(casekey, df)
        if not full_text:
            continue
        checked += 1

        for stat in d.get("reasoning", {}).get("statutes", []):
            nums = _NUM_EXTRACT_RE.findall(stat)
            if not nums:
                continue
            grounded = any(n in full_text for n in nums)
            if not grounded:
                flagged.append({
                    "case_id": case_id,
                    "statute": stat,
                    "reason": "number not found anywhere in source text",
                })

    flagged_df = pd.DataFrame(flagged)
    out_path = os.path.join(MONITORING_DIR, f"tier1_statute_flags__{checkpoint_label}.csv")
    flagged_df.to_csv(out_path, index=False)

    print(f"  [Tier 1] checked {checked} cases | {len(flagged_df)} flagged | saved -> {out_path}")
    return flagged_df


# ============================================================
# CONCEPTS / NULL-RATE DIAGNOSTIC
# ============================================================

def run_concepts_null_diagnostic(output_dir, checkpoint_label):
    null_counts = Counter()
    total = 0

    for f in glob.glob(os.path.join(output_dir, "*.json")):
        try:
            d = json.load(open(f))
        except Exception:
            continue
        total += 1
        if not d.get("reasoning", {}).get("summary"):
            null_counts["reasoning.summary"] += 1
        if not d.get("summary"):
            null_counts["summary"] += 1
        if not d.get("reasoning", {}).get("principles"):
            null_counts["principles_empty"] += 1
        if not d.get("legal_concepts"):
            null_counts["concepts_empty"] += 1
        if not d.get("citations"):
            null_counts["citations_empty"] += 1

    row = {"checkpoint": checkpoint_label, "total_cases": total}
    for k, v in null_counts.items():
        row[k] = v
        row[f"{k}_pct"] = round(100 * v / total, 1) if total else 0

    log_path = os.path.join(MONITORING_DIR, "concepts_null_diagnostic_log.csv")
    row_df = pd.DataFrame([row])
    if os.path.exists(log_path):
        row_df.to_csv(log_path, mode="a", header=False, index=False)
    else:
        row_df.to_csv(log_path, index=False)

    print(f"  [Concepts diag] total={total} | " +
          " | ".join(f"{k}={v}({row.get(f'{k}_pct','')}%)" for k, v in null_counts.items()))
    return row


# ============================================================
# TIER 2 — LLM judge on flagged cases + calibration sample
# ============================================================

TIER2_JUDGE_PROMPT = """You are auditing a legal-fact extraction pipeline against the
original Indian Supreme Court judgment text. You will be given the ORIGINAL
JUDGMENT TEXT and an EXTRACTED FIELD. Score using EXACTLY one of:
correct | partially_correct | incorrect | uncertain.

Rules:
- "incorrect" if any part is fabricated, wrongly attributed, or contradicts the source.
- Give a one-sentence justification citing the specific gap or error.
- Do not invent statute/case names not present in the source text.

FIELD_NAME: {field_name}
SOURCE_TEXT_EXCERPT: {excerpt}
EXTRACTED_VALUE: {value}

Return only JSON: {{"verdict": "...", "justification": "..."}}"""

def run_tier2_judge_on_case(case_id, filepath, df):
    d = json.load(open(filepath))
    filename = d.get("filename")
    year = d.get("year")
    casekey = f"{str(filename).lower()}__{year}"
    full_text = reconstruct_full_text(casekey, df)
    excerpt = full_text[:6000] if full_text else ""

    results = {}
    for field_name, value in [
        ("statutes", d.get("reasoning", {}).get("statutes", [])),
        ("legal_concepts", d.get("legal_concepts", [])),
    ]:
        if not value:
            continue
        prompt = TIER2_JUDGE_PROMPT.format(
            field_name=field_name, excerpt=excerpt, value=json.dumps(value, ensure_ascii=False)
        )
        try:
            raw, method = call_groq(prompt, max_tokens=300, json_schema=None, schema_name="tier2_judge")
            results[field_name] = raw
        except Exception as e:
            results[field_name] = {"verdict": "error", "justification": str(e)}
    return {"case_id": case_id, **results}


def run_tier2_full(output_dir, df, flagged_df, calibration_pct=0.02, seed=42):
    print("\n[Tier 2] Building sample (flagged cases + calibration sample)...")

    all_files = glob.glob(os.path.join(output_dir, "*.json"))
    id_to_path = {}
    for f in all_files:
        try:
            cid = json.load(open(f)).get("case_id")
            if cid:
                id_to_path[cid] = f
        except Exception:
            continue

    flagged_ids = set(flagged_df["case_id"].unique()) if len(flagged_df) else set()

    random.seed(seed)
    all_ids = list(id_to_path.keys())
    calib_n = max(1, int(len(all_ids) * calibration_pct))
    calibration_sample = set(random.sample(all_ids, min(calib_n, len(all_ids))))

    sample_ids = flagged_ids | calibration_sample
    print(f"[Tier 2] Flagged: {len(flagged_ids)} | Calibration: {len(calibration_sample)} | Total sample: {len(sample_ids)}")

    tier2_results = []
    for cid in tqdm(sample_ids, desc="Tier 2 judge"):
        fpath = id_to_path.get(cid)
        if not fpath:
            continue
        tier2_results.append(run_tier2_judge_on_case(cid, fpath, df))

    out_path = os.path.join(MONITORING_DIR, "tier2_judge_results.json")
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(tier2_results, f, indent=2, ensure_ascii=False)

    print(f"[Tier 2] Done — {len(tier2_results)} cases judged -> {out_path}")
    return tier2_results

CELL 9 — PART 1 (Helpers + Issue Hint + Beginning of process_case)

In [14]:
# ── CELL 9: MAIN LOOP — PART 1 ───────────────────────────────
from tqdm import tqdm

_ISSUE_PATTERNS = [
    r"[Tt]he (?:short |only |sole |main )?(?:question|point|issue|matter)\s+(?:that\s+)?(?:arises?|for(?:\s+our|this\s+[Cc]ourt(?:'s)?)?\s+(?:consideration|determination))\s*(?:is|was)\s*[:\-]?\s*([^.]{20,200}\.)",
    r"[Ww]hether\s+.{20,200}\?"
]


def extract_issue_hint(text):
    text = text[:5000]
    for pat in _ISSUE_PATTERNS:
        m = re.search(pat, text, re.I)
        if m:
            return m.group(0).strip()
    return None


# ============================================================
# SMART JOIN
# ============================================================

def smart_join(sentences):
    cleaned = [
        str(s).strip().rstrip(".")
        for s in sentences
        if str(s).strip()
    ]
    if not cleaned:
        return ""
    return ". ".join(cleaned) + "."


# ============================================================
# STATUTE / CITATION DEDUP HELPERS
# ============================================================

_REPORTER_CITE_RE = re.compile(
    r"\(?\d{4}\)?\s*\d+\s*(?:SCC|SCR|SCALE)\s*\d+|AIR\s*\d{4}\s*[A-Za-z\.]+\s*\d+",
    re.I
)
_CASE_NUM_CITE_RE = re.compile(r"[A-Z]{2,8}\s*\(?[A-Za-z]*\)?\s*\d+\s*/\s*\d{2,4}")

_SECTION_ALL_RE = re.compile(r"[Ss]ections?[_\s]?(\d+[A-Za-z]?(?:-[A-Za-z0-9]+)?)", re.I)
_ARTICLE_NUM_RE = re.compile(r"(?:Article|Art)[_\s\.]?(\d+[A-Za-z]?)", re.I)
_ORDER_RULE_RE  = re.compile(r"Order[_\s]?(\d+)[_\s]?Rule[_\s]?(\d+)", re.I)


_YEAR_RE = re.compile(r"^(18|19|20)\d{2}$")
_NUM_TOKEN_RE = re.compile(r"(?<!\d)(\d{1,3}[A-Za-z]?(?:\(\d+\)|\([A-Za-z]\))?)(?!\d)")

def _bare_statute_keys(s):
    low = s.lower()
    if "section" in low:
        tokens = _NUM_TOKEN_RE.findall(s)
        nums = {t for t in tokens if not _YEAR_RE.match(t)}
        if nums:
            return {f"section_{n.lower()}" for n in nums}
    if "article" in low or re.match(r"art\.?\b", low):
        m = _ARTICLE_NUM_RE.search(s)
        if m:
            return {f"article_{m.group(1).lower()}"}
    if "order" in low and "rule" in low:
        m = _ORDER_RULE_RE.search(s)
        if m:
            return {f"order_{m.group(1)}_rule_{m.group(2)}"}
    return {s.strip().lower()}


def merge_dedup_statutes(llm_list, regex_list):
    combined = list(llm_list) if llm_list else []
    seen = set()
    for c in combined:
        seen |= _bare_statute_keys(c)
    for r in (regex_list or []):
        keys = _bare_statute_keys(r)
        if not keys & seen:          # any overlap = duplicate
            combined.append(r)
            seen |= keys
    return combined


def _bare_cite_key(cite):
    """Returns a SINGLE key — citations are deduped one-to-one, not multi-value."""
    m = _REPORTER_CITE_RE.search(cite)
    if m:
        return re.sub(r"\s+", " ", m.group(0)).strip().lower()
    m = _CASE_NUM_CITE_RE.search(cite)
    if m:
        return re.sub(r"\s+", " ", m.group(0)).strip().lower()
    return cite.strip().lower()


def merge_dedup_citations(llm_list, regex_list):
    combined = list(llm_list) if llm_list else []
    seen = {_bare_cite_key(c) for c in combined}
    for r in (regex_list or []):
        k = _bare_cite_key(r)
        if k not in seen:
            combined.append(r)
            seen.add(k)
    return combined


# ============================================================
# PROCESS CASE
# ============================================================

def process_case(group: pd.DataFrame):

    filename = str(group["filename"].iloc[0])
    year = (
        int(group["year"].iloc[0])
        if pd.notna(group["year"].iloc[0])
        else 0
    )

    # =======================================================
    # BUILD LABEL MAP
    # =======================================================

    lbl_map = {}
    for _, row in group.iterrows():
        if HAS_CONFIDENCE:
            try:
                conf = float(row.get("confidence", 1.0))
            except:
                conf = 1.0
            if conf < 0.65:
                continue
        label = str(row["predicted_label"]).strip()
        txt = str(row["text"]).strip()
        lbl_map.setdefault(label, []).append(txt)

    def join(*labels):
        vals = []
        for lbl in labels:
            vals.extend(lbl_map.get(lbl, []))
        return smart_join(vals)

    # =======================================================
    # TEXT BLOCKS
    # =======================================================

    facts_text = join("Facts")
    petitioner_args_text = join("Arguments of Petitioner")
    respondent_args_text = join("Arguments of Respondent")
    reasoning_text = join("Reasoning")
    decision_text = join("Decision")

    full_text = smart_join([
        facts_text,
        petitioner_args_text,
        respondent_args_text,
        reasoning_text,
        decision_text
    ])

    # =======================================================
    # ISSUE HINT
    # =======================================================

    issue_hint = extract_issue_hint(full_text)

    # =======================================================
    # DETERMINISTIC EXTRACTION
    # =======================================================

    meta = extract_metadata(full_text)
    petitioner, respondent = extract_parties(filename, facts_text)

    # ============================================================
    # GROQ CALL 1: CASE / FACTS
    # ============================================================

    case_prompt = build_case_prompt(
        full_text=full_text,
        facts=facts_text,
        petitioner_args=petitioner_args_text,
        respondent_args=respondent_args_text,
        title_hint=title_from_filename(filename),
        year=year
    )

    case_data, method1 = call_groq(
        case_prompt,
        max_tokens=3500,
        json_schema=CASE_SCHEMA,
        schema_name="case_extraction"
    )

    # ============================================================
    # GROQ CALL 2: LEGAL ANALYSIS
    # ============================================================

    analysis_prompt = build_analysis_prompt(
        facts=facts_text,
        reasoning=reasoning_text,
        decision=decision_text,
        full_text=full_text
    )

    analysis_data, method2 = call_groq(
        analysis_prompt,
        max_tokens=3000,
        json_schema=ANALYSIS_SCHEMA,
        schema_name="legal_analysis"
    )

    # ============================================================
    # MERGE BOTH GROQ RESULTS
    # ============================================================

    sem = {}
    if isinstance(case_data, dict):
        sem.update(case_data)
    if isinstance(analysis_data, dict):
        sem.update(analysis_data)

    extraction_method = f"case:{method1}|analysis:{method2}"

    statutes = merge_dedup_statutes(sem.get("statutes") or [], extract_statutes(full_text))
    citations = merge_dedup_citations(sem.get("citations") or [], extract_citations(full_text))
    concepts = sem.get("legal_concepts") or []   # LLM-only, constrained by ANALYSIS_SCHEMA enum

    parties = sem.get("parties") or {}
    petitioner = parties.get("petitioner")
    respondent = parties.get("respondent")

    # =======================================================
    # OUTCOME NORMALIZATION
    # =======================================================

    groq_outcome = normalize_outcome(sem.get("outcome"))
    final_outcome = groq_outcome or regex_outcome(decision_text) or "Unknown"
    sem["outcome"] = final_outcome

    # =======================================================
    # OUTPUT RECORD
    # =======================================================

    record = {

        "extraction_method": extraction_method,
        "case_id": make_case_id(year, filename),
        "year": year,
        "title": sem.get("title") or title_from_filename(filename),
        "filename": filename,

        "metadata": {
            "date": extract_date_from_filename(filename) or meta["date"],
            "court": sem.get("court") or meta["court"],
            "case_type": sem.get("case_type"),
        },

        "parties": {
            "petitioner": petitioner,
            "respondent": respondent,
        },

        "issue": sem.get("issue"),

        "facts": sem.get("facts") or [],

        "petitioner_arguments": sem.get("petitioner_arguments") or [],
        "respondent_arguments": sem.get("respondent_arguments") or [],

        "reasoning": {
            "statutes": statutes,
            "principles": sem.get("reasoning_principles") or [],
            "summary": sem.get("reasoning_summary"),
        },

        "decision": {
            "outcome": final_outcome,
            "details": sem.get("decision_details"),
            "outcome_reasoning": sem.get("outcome_reasoning"),
        },

        "legal_concepts": concepts,
        "citations": citations,

        "summary": sem.get("case_summary"),
    }

    return record


# ============================================================
# RUN
# ============================================================

done_keys = load_done()

# ==========================================
# FILTER: 2000–2025 cases only, split into 4 quarters
# ==========================================

YEAR_MIN = 2000
YEAR_MAX = 2025

QUARTER = 3   # <-- CHANGE THIS PER PERSON: 1, 2, 3, or 4

year_lookup = (
    df.drop_duplicates(subset="_casekey")
      .set_index("_casekey")["year"]
)

filtered_keys = [
    k for k in all_keys
    if pd.notna(year_lookup.get(k))
    and YEAR_MIN <= year_lookup.get(k) <= YEAR_MAX
]

quarter_size = len(filtered_keys) // 4
q_start = (QUARTER - 1) * quarter_size
q_end = (
    quarter_size * QUARTER
    if QUARTER < 4
    else len(filtered_keys)
)

quarter_keys = filtered_keys[q_start:q_end]

todo = [k for k in quarter_keys if k not in done_keys]

print(f"📊 2000–2025 cases total: {len(filtered_keys):,}")
print(f"🧪 Quarter {QUARTER} target: {len(quarter_keys):,} cases")
print(f"⏳ Remaining: {len(todo):,} cases")

session_done = 0
errors = []
show_first_err = True
stop_processing = False
MONITORING_INTERVAL = 500
next_monitor_threshold = ((len(done_keys) // MONITORING_INTERVAL) + 1) * MONITORING_INTERVAL
for batch_start in range(0, len(todo), BATCH_SIZE):

    if stop_processing:
        break

    if session_done >= DAILY_LIMIT:
        print("\nDaily case limit reached.")
        break

    batch_keys = todo[batch_start:batch_start + BATCH_SIZE]
    batch_df = df[df["_casekey"].isin(set(batch_keys))].copy()

    for key in tqdm(
        batch_keys,
        desc=f"Batch {batch_start // BATCH_SIZE + 1}",
        leave=False
    ):
        try:
            group = batch_df[batch_df["_casekey"] == key]

            if group.empty:
                errors.append({"key": key, "error": "empty group"})
                continue

            record = process_case(group)
            save_case(record)

            if "regex_fallback" in record["extraction_method"]:
                # at least one call degraded — don't checkpoint as done, log for retry
                errors.append({"key": key, "error": f"partial/full LLM failure: {record['extraction_method']}"})
            else:
                done_keys.add(key)
                session_done += 1

        except DailyLimitReached:
            print("\nDaily token limit reached.")
            save_checkpoint_and_stop(done_keys, errors)
            stop_processing = True
            break

        except KeyboardInterrupt:
            print("\nInterrupted.")
            save_checkpoint_and_stop(done_keys, errors)
            raise

        except Exception:
            tb = traceback.format_exc()
            errors.append({"key": key, "error": tb})
            if show_first_err:
                print(f"\nFirst error ({key}):\n{tb}")
                show_first_err = False

    save_done(done_keys)
    save_errors(errors)
    del batch_df
    gc.collect()

    remaining = len(todo) - session_done

    print(
        f"\nBatch {batch_start // BATCH_SIZE + 1} done\n"
        f"Session Cases: {session_done:,}\n"
        f"Global Cases: {len(done_keys):,}\n"
        f"Remaining: {remaining:,}\n"
        f"Errors: {len(errors):,}\n"
        f"Tokens Used: {total_tokens_used:,} / {MAX_TOKEN_BUDGET:,}"
    )

        # ── Auto-run Tier 1 + concepts diagnostic every 500 cases ──
    while len(done_keys) >= next_monitor_threshold:
        checkpoint_label = f"{next_monitor_threshold}cases"
        print(f"\n🔍 Monitoring checkpoint reached: {next_monitor_threshold} cases")
        run_tier1_audit(OUTPUT_DIR, df, checkpoint_label)
        run_concepts_null_diagnostic(OUTPUT_DIR, checkpoint_label)
        next_monitor_threshold += MONITORING_INTERVAL

# ============================================================
# FINAL SUMMARY
# ============================================================

print(f"\nDone — {session_done:,} JSON files written to {OUTPUT_DIR}")

if errors:
    err_path = os.path.join(CHECKPOINT_DIR, "errors.json")
    print(f"{len(errors):,} errors logged -> {err_path}")

# ============================================================
# END-OF-RUN: FINAL TIER 1 + TIER 2 + CONCEPTS DIAGNOSTIC
# ============================================================

print("\n" + "="*60)
print("RUN COMPLETE — running final monitoring pass")
print("="*60)

final_flagged_df = run_tier1_audit(OUTPUT_DIR, df, "FINAL")
run_concepts_null_diagnostic(OUTPUT_DIR, "FINAL")
run_tier2_full(OUTPUT_DIR, df, final_flagged_df, calibration_pct=0.02)

print("\n✅ All monitoring artifacts saved to:", MONITORING_DIR)

↩ Resuming — 2,487 cases already completed
📊 2000–2025 cases total: 9,986
🧪 Quarter 3 target: 2,496 cases
⏳ Remaining: 9 cases



Batch 1 done
Session Cases: 9
Global Cases: 2,496
Remaining: 0
Errors: 0
Tokens Used: 82,315 / 480,000

Done — 9 JSON files written to /content/drive/MyDrive/fact_layer_v_n_final

RUN COMPLETE — running final monitoring pass
  [Tier 1] checked 2491 cases | 40 flagged | saved -> /content/drive/MyDrive/fact_layer_v_n_final/monitoring/tier1_statute_flags__FINAL.csv
  [Concepts diag] total=2496 | reasoning.summary=943(37.8%) | summary=455(18.2%) | principles_empty=1206(48.3%) | citations_empty=216(8.7%) | concepts_empty=374(15.0%)

[Tier 2] Building sample (flagged cases + calibration sample)...
[Tier 2] Flagged: 26 | Calibration: 49 | Total sample: 74


Tier 2 judge: 100%|██████████| 74/74 [20:47<00:00, 16.86s/it]


[Tier 2] Done — 74 cases judged -> /content/drive/MyDrive/fact_layer_v_n_final/monitoring/tier2_judge_results.json

✅ All monitoring artifacts saved to: /content/drive/MyDrive/fact_layer_v_n_final/monitoring


In [15]:
import time, openai
c = client.with_options(timeout=30.0, max_retries=0)

def ping(model, stream=False):
    t = time.time()
    try:
        r = c.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": "Reply with the word OK."}],
            max_tokens=300, stream=stream,
        )
        if stream:
            next(iter(r))   # wait for the first chunk
            print(f"{model} [stream]: first chunk in {time.time()-t:.1f}s")
        else:
            print(f"{model}: OK in {time.time()-t:.1f}s")
    except openai.APIStatusError as e:
        print(f"{model}: HTTP {e.status_code} after {time.time()-t:.1f}s -> {str(e)[:200]}")
    except Exception as e:
        print(f"{model}: {type(e).__name__} after {time.time()-t:.1f}s")

ids = [m.id for m in c.models.list().data]
print(ids)   # check whether meta/llama-3.1-8b-instruct is even listed

ping("openai/gpt-oss-20b", stream=True)

small = [i for i in ids if any(k in i.lower() for k in ("8b", "nano", "mini", "gemma", "3b", "4b"))][:4]
for m in small:
    ping(m)

['01-ai/yi-large', 'adept/fuyu-8b', 'ai21labs/jamba-1.5-large-instruct', 'aisingapore/sea-lion-7b-instruct', 'bigcode/starcoder2-15b', 'databricks/dbrx-instruct', 'deepseek-ai/deepseek-coder-6.7b-instruct', 'deepseek-ai/deepseek-v4.1-flash', 'google/codegemma-1.1-7b', 'google/codegemma-7b', 'google/deplot', 'google/diffusiongemma-26b-a4b-it', 'google/gemma-2b', 'google/gemma-3-12b-it', 'google/gemma-3-4b-it', 'google/gemma-4-31b-it', 'google/recurrentgemma-2b', 'ibm/granite-3.0-3b-a800m-instruct', 'ibm/granite-3.0-8b-instruct', 'ibm/granite-34b-code-instruct', 'ibm/granite-8b-code-instruct', 'meta/codellama-70b', 'meta/llama-3.2-11b-vision-instruct', 'meta/llama-3.2-90b-vision-instruct', 'meta/llama-guard-4-12b', 'meta/llama2-70b', 'meta/muse-glimmer-30b', 'microsoft/kosmos-2', 'microsoft/phi-3-vision-128k-instruct', 'microsoft/phi-3.5-moe-instruct', 'mistralai/codestral-22b-instruct-v0.1', 'mistralai/mistral-7b-instruct-v0.3', 'mistralai/mistral-large', 'mistralai/mistral-large-2-inst

**Verfication beyond this(DON'T EXECUTE)**

In [16]:
target_case_id = "2003_079a48222227"

# reverse-lookup: find the _casekey whose mapped case_id matches
target_casekey = next(
    k for k, cid in casekey_to_caseid.items() if cid == target_case_id
)

group = df[df["_casekey"] == target_casekey].copy()
record = process_case(group)

import json
print(json.dumps(record["reasoning"]["statutes"], indent=2, ensure_ascii=False))

NameError: name 'casekey_to_caseid' is not defined

In [ ]:
test_strings = [
    "Contempt of Courts Act, 1971 Sections 2, 3, 79, 80",
    "Code of Civil Procedure, 1908 Sections 76, 92",
    "Act Sections 18, 19, 70, 72",
    "Section_92", "Section_19", "Section_79", "Section_80",
    "Section_70", "Section_72",
]
for s in test_strings:
    print(f"{s!r:60} -> {_bare_statute_keys(s)}")

In [ ]:
test_strings = [
    "Section 69(2) of the Partnership Act, 1932",
    "Section_69(2)",
    "Section 5(2) of the Act",
    "Section_5(2)",
    "Sections 2, 3, 79, 80",
]
for s in test_strings:
    print(f"{s!r:50} -> {_bare_statute_keys(s)}")

In [ ]:
# ── CELL 10: Verify — print one sample file ──────────────────

import glob

files = glob.glob(
    os.path.join(
        OUTPUT_DIR,
        "*.json"
    )
)

print(
    f"Total JSON files in output folder: "
    f"{len(files):,}\n"
)

if not files:

    print(
        "⚠️ No JSON files found."
    )

else:

    sample_file = files[0]

    print(
        f"Sample file:\n{sample_file}\n"
    )

    with open(
        sample_file,
        encoding="utf-8"
    ) as f:

        sample = json.load(f)

    print(
        json.dumps(
            sample,
            indent=2,
            ensure_ascii=False
        )
    )

    print(
        "\n=================================="
    )

    print(
        "Extraction Method:",
        sample.get(
            "extraction_method"
        )
    )

    print(
        "Court:",
        sample.get(
            "metadata",
            {}
        ).get(
            "court"
        )
    )

    print(
        "Case Type:",
        sample.get(
            "metadata",
            {}
        ).get(
            "case_type"
        )
    )

    print(
        "Outcome:",
        sample.get(
            "decision",
            {}
        ).get(
            "outcome"
        )
    )

    print(
        "Outcome Reasoning:",
        sample.get(
            "decision",
            {}
        ).get(
            "outcome_reasoning"
        )
    )

    print(
        "\n=================================="
    )

    print(
        f"Total Tokens Used: "
        f"{total_tokens_used:,}"
    )

In [ ]:
import json, glob
from collections import Counter

null_counts = Counter()
total = 0
for f in glob.glob(os.path.join(OUTPUT_DIR, "*.json")):
    d = json.load(open(f))
    total += 1
    if not d["reasoning"]["summary"]:
        null_counts["reasoning.summary"] += 1
    if not d["summary"]:
        null_counts["summary"] += 1
    if not d["reasoning"]["principles"]:
        null_counts["principles_empty"] += 1
    if not d["legal_concepts"]:
        null_counts["concepts_empty"] += 1
    if not d["citations"]:
        null_counts["citations_empty"] += 1

for k, v in null_counts.items():
    print(f"{k}: {v}/{total} ({100*v/total:.1f}%)")

In [ ]:
import json, glob
from collections import Counter

outcome_when_null_summary = Counter()
outcome_when_have_summary = Counter()

for f in glob.glob(os.path.join(OUTPUT_DIR, "*.json")):
    d = json.load(open(f))
    outcome = d["decision"]["outcome"]
    if not d["reasoning"]["summary"]:
        outcome_when_null_summary[outcome] += 1
    else:
        outcome_when_have_summary[outcome] += 1

print("Outcome distribution WHEN reasoning.summary is null:")
for k, v in outcome_when_null_summary.items():
    print(f"  {k}: {v}")

print("\nOutcome distribution WHEN reasoning.summary is present:")
for k, v in outcome_when_have_summary.items():
    print(f"  {k}: {v}")

In [ ]:
import json, glob, os

unknowns = []
for f in glob.glob(os.path.join(OUTPUT_DIR, "*.json")):
    d = json.load(open(f))
    if d["decision"]["outcome"] == "Unknown":
        unknowns.append(f)

print(f"{len(unknowns)} Unknown-outcome cases:")
for f in unknowns:
    print(" ", os.path.basename(f))

In [ ]:
import json, glob, os

unknowns = []
for f in glob.glob(os.path.join(OUTPUT_DIR, "*.json")):
    d = json.load(open(f))
    if d["decision"]["outcome"] == "Directions Issued":
        unknowns.append(f)

print(f"{len(unknowns)} Unknown-outcome cases:")
for f in unknowns:
    print(" ", os.path.basename(f))

In [ ]:
import json, glob
import pandas as pd

rows = []
for f in glob.glob(os.path.join(OUTPUT_DIR, "*.json")):
    d = json.load(open(f))
    rows.append({
        "outcome": d["decision"]["outcome"],
        "summary_null": not d["reasoning"]["summary"]
    })

df_diag = pd.DataFrame(rows)

print("=== Avg reasoning_text length by outcome ===")
print(df_diag.groupby("outcome")["reasoning_len"].mean())

print("\n=== Avg reasoning_text length: null-summary vs has-summary ===")
print(df_diag.groupby("summary_null")["reasoning_len"].mean())

print("\n=== % null-summary by outcome ===")
print(df_diag.groupby("outcome")["summary_null"].mean() * 100)

In [ ]:
# ============================================================
# SPOT CHECK: Mediwell Hospital — confirm Directions Issued fix
# ============================================================

target_case_id = "1996_80ae8cd2b5f6"

# reverse-lookup: find the _casekey whose mapped case_id matches
target_casekey = next(
    k for k, cid in casekey_to_caseid.items() if cid == target_case_id
)

group = df[df["_casekey"] == target_casekey].copy()
record = process_case(group)

import json
print(json.dumps({
    "case_id": record["case_id"],
    "title": record["title"],
    "outcome": record["decision"]["outcome"],
    "outcome_reasoning": record["decision"]["outcome_reasoning"],
    "decision_details": record["decision"]["details"],
    "extraction_method": record["extraction_method"],
}, indent=2, ensure_ascii=False))

In [ ]:
import json

def spot_check(case_id):
    """Run process_case() fresh on one case_id and return key fields."""
    casekey = next(
        k for k, cid in casekey_to_caseid.items() if cid == case_id
    )
    group = df[df["_casekey"] == casekey].copy()
    record = process_case(group)
    return record

CHECK_CASES = {
    "Jagdish & Anr murder case (Art 7/9/14 false-positive statutes)": "2000_6c941e2104ed"

}

results = {}

for label, case_id in CHECK_CASES.items():
    print(f"\n{'='*70}\n{label}  ({case_id})\n{'='*70}")
    try:
        record = spot_check(case_id)
        results[case_id] = record

        print("outcome:          ", record["decision"]["outcome"])
        print("legal_concepts:   ", record["legal_concepts"])
        print("statutes:         ", record["reasoning"]["statutes"])
        print("extraction_method:", record["extraction_method"])
    except StopIteration:
        print(f"⚠️ case_id {case_id} not found in casekey_to_caseid map — check it's spelled correctly / exists in this CSV")
    except Exception as e:
        print(f"⚠️ Error processing {case_id}: {e}")
